# Goal 2：真实四模式、搜索预算与冻结记忆

本工作 Notebook 对应 LLM 作业的四模式和示范记忆对照。生产 A/B/C 调度与被评估系统的模型调用单独计数。默认复算使用已经锁定的真实提议，身份是 RECOMPUTE，不伪装为新 LIVE。

In [ ]:
from pathlib import Path
import sys, json, tempfile, shutil
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'AGENTS.md').is_file() and (p/'task1').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from task1.workflow.io import read_json, digest, object_hash
EV = ROOT/'task1/evidence/goal2'
current = read_json(EV/'current_runs.json')
contract = read_json(ROOT/'task1/config/goal2/contract.json')
split = read_json(EV/'data/split_manifest.json')
MODE = 'RECOMPUTE'
ENABLE_LIVE = False
assert MODE == 'RECOMPUTE' and ENABLE_LIVE is False, '新的 LIVE 调用须使用独立命令并显式启用，默认 Notebook 不发起模型请求。'
WORK = Path(tempfile.mkdtemp(prefix='sc-g2-notebook-'))
def show(rows):
    display(pd.DataFrame(rows))
def figure(name):
    directory = Path(recomputed.get('figure_directory', WORK/'figures'))
    target = directory/(name+'.png')
    assert target.is_file(), f'复算图缺失: {target}'
    display(Image(filename=str(target), width=1050))
print('运行方式:', MODE, '| 新模型调用:', 0)
print('原始来源 datum:', contract['source_crs'], '| 单位: 工作米 / 原始相对秒')
print('当前固定分区:', {name:len(ids) for name,ids in split['splits'].items()})


## 四种决策路径

| 模式 | 决策模型 | 搜索反馈 | 示范记忆 |
|---|---|---|---|
| llm-only | 一次真实锁定提议 | 无 | 无 |
| search-only | 严格 0 次；不实例化 Provider | 确定性至多 20 候选 | 无 |
| llm+search | 至多 3 回合真实决策 | 仅本 episode 已执行结果 | 无 |
| llm+memory+search | 相同模型、回合、预算 | 同上 | 冻结 DEMO_MEMORY，只读 |

一次调用至多 8 张诊断卡，调用数按实际批次算。LLM-only 的计算预算不同是结构消融的一部分，不伪称等量计算。

## 从 raw 与已保存真实提议重建确定性执行

复算重新处理开发和三次阶段评测中每个实际候选，检查原始提议合法性、预算、选择与锁定后评价；不用固定 Mock 答案。Provider 哨兵阻断任何意外模型调用，持久记忆保持原哈希。

In [ ]:
from task1.scripts.goal2 import recompute
recomputed = recompute(topic='modes', output_directory=WORK, rebuild_figures=True)
assert recomputed['status'] == 'VERIFIED', recomputed
assert recomputed['new_model_calls'] == 0, recomputed
print('从 raw 和冻结提议复算:', recomputed['status'])
print('新增模型调用:', recomputed['new_model_calls'])
print('检查项数量:', len(recomputed['checks']))
show(recomputed['checks'][:20])
print('全部检查结果保存在独立复算输出，不把重复计算计为新独立实验。')


In [ ]:
mode_run = EV/'runs'/current['mode_evaluation']
mode_manifest = read_json(mode_run/'manifest.json')
episodes = []
for entry in mode_manifest['mode_episodes']:
    path = mode_run/entry['path']
    assert digest(path) == entry['sha256']
    episodes.append(read_json(path))
mode_rows = [row for episode in episodes for row in episode['records']]
show([{'mode':episode['mode'],'episode':episode['episode'],'records':len(episode['input_ids']),'classification':episode['classification'],'model_dispatches':episode['resources']['experiment_model_dispatches'],'candidate_evaluations':episode['resources']['candidate_evaluations'],'elapsed_seconds':episode['elapsed_seconds']} for episode in episodes])
assert all(episode['resources']['experiment_model_dispatches']==0 for episode in episodes if episode['mode']=='search-only')

## 逐记录配对、失效动作与成本

同一原始记录的三 episode 保留为重复决策，不能当作三个独立样本。图中描述相对该记录 reference 的有符号变化；空误差不填 0。原始非法提议和工程回退分别显示，回退不能替原始建议取得合法性分数。

In [ ]:
figure('four_mode_paired_results')
figure('four_mode_failures_and_cost')
proposal_rows=[]
for row in mode_rows:
    for proposal in row['proposal_records']:
        proposal_rows.append({'record_id':row['record_id'],'mode':row['mode'],'episode':row['episode'],'round':proposal['round'],'legal':proposal['legal'],'executed':proposal['executed'],'rejection_reason':proposal.get('rejection_reason',proposal.get('reason')),'original':proposal.get('original')})
show(proposal_rows[:12])
print('全部原始提议条目:',len(proposal_rows),'；这里只展示前12条，完整来源保留。')

## 检索、消费与泄漏隔离

DEMO 60 条先经过确定性核验，再冻结快照。检索按父记录/同源组、分区、版本和原始诊断适用性过滤；评测不写记忆、不读取其他模式结果。检索命中、模型引用、参数与案例一致是不同级别的可观察事实，并不单独证明因果收益。

In [ ]:
figure('memory_coverage_and_consumption')
memory_rows = [row for row in mode_rows if row['mode']=='llm+memory+search']
hashes = sorted({row['memory_snapshot_hash'] for row in memory_rows})
assert len(hashes)==1 and hashes[0], hashes
print('全部正式记忆episode使用同一snapshot:',hashes[0])
show([{'record_id':row['record_id'],'episode':row['episode'],'round':rd['round'],**rd['memory_consumption']} for row in memory_rows for rd in row['rounds'] if 'memory_consumption' in rd][:12])

## 真实调用与理解边界

原始请求、响应、provider/session ID 和可见 tokens 在各 round 的调用目录；底层不可见请求/费用保持 unknown。实际预测只和事前绑定的 metric/reference/sign 比较。新的 LIVE 调用使用 `python -m task1.scripts.goal2 --help` 中明确的 LIVE 入口；改变本 Notebook 的显示字符串不会启动模型。

In [ ]:
print('本 Notebook 新模型调用:',recomputed['new_model_calls'])
print('先前真实正式派发:',sum(episode['resources']['experiment_model_dispatches'] for episode in episodes))
print('GPT_SECOND_REVIEW=PENDING; Submission=NOT_READY')
shutil.rmtree(WORK)